Tabular urinary biomarkers for pancreatic cancer: https://www.kaggle.com/datasets/johnjdavisiv/urinary-biomarkers-for-pancreatic-cancer

In [2]:
import pandas as pd

df = pd.read_csv("../data/raw/Debernardi et al 2020 data.csv")
df

,sample_id,patient_cohort,sample_origin,age,sex,diagnosis,stage,benign_sample_diagnosis,plasma_CA19_9,creatinine,LYVE1,REG1B,TFF1,REG1A
0,S1,Cohort1,BPTB,33,F,1,NaN,NaN,11.7,1.83222,0.893219,52.948840,654.282174,1262.000
1,S10,Cohort1,BPTB,81,F,1,NaN,NaN,NaN,0.97266,2.037585,94.467030,209.488250,228.407
2,S100,Cohort2,BPTB,51,M,1,NaN,NaN,7.0,0.78039,0.145589,102.366000,461.141000,NaN
3,S101,Cohort2,BPTB,61,M,1,NaN,NaN,8.0,0.70122,0.002805,60.579000,142.950000,NaN
4,S102,Cohort2,BPTB,62,M,1,NaN,NaN,9.0,0.21489,0.000860,65.540000,41.088000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
585,S549,Cohort2,BPTB,68,M,3,IV,NaN,NaN,0.52026,7.058209,156.241000,525.178000,NaN
586,S558,Cohort2,BPTB,71,F,3,IV,NaN,NaN,0.85956,8.341207,16.915000,245.947000,NaN
587,S560,Cohort2,BPTB,63,M,3,IV,NaN,NaN,1.36851,7.674707,289.701000,537.286000,NaN
588,S583,Cohort2,BPTB,75,F,3,IV,NaN,NaN,1.33458,8.206777,205.930000,722.523000,NaN


print(df.info())
print(df.describe())
print(df.isnull().sum())

In [4]:
#null values
print(df.groupby('diagnosis')['stage'].count())          
print(df.groupby('diagnosis')['benign_sample_diagnosis'].count())  

diagnosis
1      0
2      0
3    199
Name: stage, dtype: int64
diagnosis
1      0
2    208
3      0
Name: benign_sample_diagnosis, dtype: int64


In [5]:
df = df.drop(columns=['stage', 'benign_sample_diagnosis'])

diagnosis -> 1 = control (no pancreatic disease), 2 = benign hepatobiliary disease (119 of which are chronic pancreatitis); 3 = cancer

aligned with diagnosis class by design — stage only exists for cancer patients, benign_sample_diagnosis only for benign. Zero ambiguity. Drop both. The only reason to keep them would be to engineer a presence flag, but that flag would directly encode diagnosis, leading to leakage.

#### CA 19
Healthy controls (diagnosis=1) have essentially 0% coverage because CA19-9 wasn't ordered — it's a cancer marker. The missingness itself carries signal. If we impute without flagging, we'll be giving healthy patients a fake CA19-9 value, which will hurt the model in the future. We create a **CA19_9_observed (1/0)**, then impute the missing values with group-level medians just to fill the gap.

In [7]:
#plasma ca19
df['CA19_9_observed'] = df['plasma_CA19_9'].notna().astype(int)
df['plasma_CA19_9'] = df.groupby('diagnosis')['plasma_CA19_9'].transform(
    lambda x: x.fillna(x.median())
)
df.isnull().sum()

sample_id            0
patient_cohort       0
sample_origin        0
age                  0
sex                  0
diagnosis            0
plasma_CA19_9        0
creatinine           0
LYVE1                0
REG1B                0
TFF1                 0
REG1A              284
CA19_9_observed      0
dtype: int64

#### REG1A

REG1A was only measured in one cohort but covered all diagnosis classes, so rather than dropping it we impute the missing values using the diagnosis group mean from the cohort where measurements appeared. This preserves the biological signal that differs across diagnosis classes while avoiding the mistake of imputing a single global mean that ignores disease status.

In [9]:
df.groupby('patient_cohort')['REG1A'].count() 

patient_cohort
Cohort1    306
Cohort2      0
Name: REG1A, dtype: int64

In [10]:
diagnosis_means = df.groupby('diagnosis')['REG1A'].mean()
df['REG1A'] = df.apply(
    lambda row: diagnosis_means[row['diagnosis']] 
    if pd.isna(row['REG1A']) 
    else row['REG1A'],
    axis=1
)
df.isnull().sum()

sample_id          0
patient_cohort     0
sample_origin      0
age                0
sex                0
diagnosis          0
plasma_CA19_9      0
creatinine         0
LYVE1              0
REG1B              0
TFF1               0
REG1A              0
CA19_9_observed    0
dtype: int64

#### Define features and target - encode what's needed

- Drop: sample id
- Encode: patient cohort, sex, sample origin

In [12]:
if 'sample_id' in df.columns:
    df = df.drop(columns=['sample_id'])
    
df['sex'] = df['sex'].map({'M': 1, 'F': 0})
df['patient_cohort'] = df['patient_cohort'].map({'Cohort1': 0, 'Cohort2': 1})

df.info()
df

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 590 entries, 0 to 589
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   patient_cohort   590 non-null    int64  
 1   sample_origin    590 non-null    object 
 2   age              590 non-null    int64  
 3   sex              590 non-null    int64  
 4   diagnosis        590 non-null    int64  
 5   plasma_CA19_9    590 non-null    float64
 6   creatinine       590 non-null    float64
 7   LYVE1            590 non-null    float64
 8   REG1B            590 non-null    float64
 9   TFF1             590 non-null    float64
 10  REG1A            590 non-null    float64
 11  CA19_9_observed  590 non-null    int64  
dtypes: float64(6), int64(5), object(1)
memory usage: 55.4+ KB


,patient_cohort,sample_origin,age,sex,diagnosis,plasma_CA19_9,creatinine,LYVE1,REG1B,TFF1,REG1A,CA19_9_observed
0,0,BPTB,33,0,1,11.700000,1.83222,0.893219,52.948840,654.282174,1262.000000,1
1,0,BPTB,81,0,1,5.334598,0.97266,2.037585,94.467030,209.488250,228.407000,0
2,1,BPTB,51,1,1,7.000000,0.78039,0.145589,102.366000,461.141000,227.871886,1
3,1,BPTB,61,1,1,8.000000,0.70122,0.002805,60.579000,142.950000,227.871886,1
4,1,BPTB,62,1,1,9.000000,0.21489,0.000860,65.540000,41.088000,227.871886,1
...,...,...,...,...,...,...,...,...,...,...,...,...
585,1,BPTB,68,1,3,427.500000,0.52026,7.058209,156.241000,525.178000,1138.323721,0
586,1,BPTB,71,0,3,427.500000,0.85956,8.341207,16.915000,245.947000,1138.323721,0
587,1,BPTB,63,1,3,427.500000,1.36851,7.674707,289.701000,537.286000,1138.323721,0
588,1,BPTB,75,0,3,427.500000,1.33458,8.206777,205.930000,722.523000,1138.323721,0


In [13]:
print(df['sample_origin'].value_counts())
print(f"\nUnique values: {df['sample_origin'].nunique()}")
print(df['sample_origin'].unique())

sample_origin
BPTB    409
LIV     132
ESP      29
UCL      20
Name: count, dtype: int64

Unique values: 4
['BPTB' 'LIV' 'ESP' 'UCL']


In [14]:
df['sample_origin'] = df['sample_origin'].map({'BPTB': 1, 'LIV': 2, 'ESP': 3, 'UCL': 4})
df = df.astype('float64')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 590 entries, 0 to 589
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   patient_cohort   590 non-null    float64
 1   sample_origin    590 non-null    float64
 2   age              590 non-null    float64
 3   sex              590 non-null    float64
 4   diagnosis        590 non-null    float64
 5   plasma_CA19_9    590 non-null    float64
 6   creatinine       590 non-null    float64
 7   LYVE1            590 non-null    float64
 8   REG1B            590 non-null    float64
 9   TFF1             590 non-null    float64
 10  REG1A            590 non-null    float64
 11  CA19_9_observed  590 non-null    float64
dtypes: float64(12)
memory usage: 55.4 KB


In [20]:
df.to_csv('../data/processed/dataset1_processed.csv', index=False)